
# Notebook 10 · Segmentación de valor omnicanal y CLV proxy

## Pregunta de negocio
¿Qué segmentos aportan más margen neto esperado y menor dependencia promocional?

## Qué aprenderás
- Cómo pasar de ingresos a margen neto.
- Cómo construir un CLV proxy sencillo y accionable.
- Cómo segmentar clientes desde una lógica de marketing de precisión.

## Outputs
- `nb10_customer_value_segmentation.csv`
- `nb10_segment_summary.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

fact = tables["fact"].copy()
costes = tables["costes"].copy()
cupones = tables["cupones"].copy()
crm = tables["crm"].copy()

sales = fact[(fact["tipo_movimiento"] == "VENTA") & fact["customer_id"].notna()].copy()

channel_pivot = (
    sales.groupby(["customer_id", "canal_movimiento"])["importe_abs_num"]
         .sum()
         .unstack(fill_value=0)
         .rename(columns=lambda c: f"revenue_{str(c).lower()}")
         .reset_index()
)

cust_cost = (
    costes[costes["customer_id"].notna()]
        .groupby("customer_id")
        .agg(
            gross_revenue=("ingreso_bruto_eur", "sum"),
            net_margin=("margen_contribucion_neto_eur", "sum"),
            discount_total=("descuento_total_eur", "sum"),
            cac_total=("cac_asignado_eur", "sum"),
            expected_return_cost=("coste_devolucion_esperado_eur", "sum"),
            tx_count=("transaction_cost_id", "nunique"),
            first_tx=("fecha_transaccion_dt", "min"),
            last_tx=("fecha_transaccion_dt", "max"),
        )
        .reset_index()
)

coupon_stats = (
    cupones.groupby("customer_id")
           .agg(
               coupons=("coupon_instance_id", "nunique"),
               redeemed=("estado_cupon", lambda s: s.astype(str).str.lower().eq("canjeado").sum()),
               avg_coupon_discount=("descuento_pct", "mean"),
           )
           .reset_index()
)

customer_value = (
    cust_cost.merge(channel_pivot, on="customer_id", how="left")
             .merge(coupon_stats, on="customer_id", how="left")
             .merge(crm[["customer_id", "tier_fidelizacion", "canal_preferido_declarado", "consentimiento_marketing"]], on="customer_id", how="left")
)

for c in ["revenue_online", "revenue_pos", "coupons", "redeemed", "avg_coupon_discount"]:
    if c in customer_value.columns:
        customer_value[c] = customer_value[c].fillna(0)

customer_value["omnichannel_flag"] = ((customer_value["revenue_online"] > 0) & (customer_value["revenue_pos"] > 0)).astype(int)
customer_value["online_share"] = customer_value["revenue_online"] / (customer_value["revenue_online"] + customer_value["revenue_pos"]).replace(0, np.nan)
customer_value["discount_rate"] = customer_value["discount_total"] / customer_value["gross_revenue"].replace(0, np.nan)
customer_value["coupon_redemption_rate"] = customer_value["redeemed"] / customer_value["coupons"].replace(0, np.nan)

ref_date = customer_value["last_tx"].max() + pd.Timedelta(days=1)
customer_value["recency_days"] = (ref_date - customer_value["last_tx"]).dt.days
customer_value["active_months"] = ((customer_value["last_tx"] - customer_value["first_tx"]).dt.days / 30.4).clip(lower=1)
customer_value["margin_per_tx"] = customer_value["net_margin"] / customer_value["tx_count"].replace(0, np.nan)
customer_value["tx_per_year"] = customer_value["tx_count"] / customer_value["active_months"] * 12
customer_value["retention_factor"] = (
    1.05
    - customer_value["recency_days"].fillna(365) / 365
    + 0.10 * customer_value["omnichannel_flag"]
    - 0.35 * customer_value["discount_rate"].fillna(0)
).clip(lower=0.20, upper=1.20)
customer_value["clv_proxy_12m"] = customer_value["margin_per_tx"] * customer_value["tx_per_year"] * customer_value["retention_factor"]

customer_value[["customer_id", "gross_revenue", "net_margin", "clv_proxy_12m", "discount_rate", "coupon_redemption_rate", "recency_days"]].head(10)


,customer_id,gross_revenue,net_margin,clv_proxy_12m,discount_rate,coupon_redemption_rate,recency_days
0,C0001,5493.16,1493.58,869.757581,0.209721,NaN,2.0
1,C0002,13579.05,3139.63,3586.047784,0.300084,0.166667,2.0
2,C0003,6343.99,1962.32,11326.467598,0.173916,0.000000,103.0
3,C0004,10791.38,5537.47,2794.119519,0.113988,NaN,126.0
4,C0005,18529.45,7405.59,4221.503750,0.054590,NaN,66.0
5,C0006,16481.18,8080.13,12567.190947,0.052542,NaN,38.0
6,C0007,14687.13,4487.45,3313.954481,0.058719,NaN,96.0
7,C0008,3908.58,1448.75,2088.586244,0.041000,0.000000,64.0
8,C0009,5400.36,985.89,1427.690618,0.194826,0.200000,8.0
9,C0010,5645.47,2442.43,31772.923499,0.102291,NaN,11.0


In [3]:

seg_cols = ["net_margin", "clv_proxy_12m", "discount_rate", "coupon_redemption_rate", "recency_days", "omnichannel_flag"]
seg_matrix = customer_value[seg_cols].copy().fillna({
    "discount_rate": 0,
    "coupon_redemption_rate": 0,
    "recency_days": customer_value["recency_days"].median(),
    "net_margin": 0,
    "clv_proxy_12m": 0,
    "omnichannel_flag": 0,
})

X = StandardScaler().fit_transform(seg_matrix)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
customer_value["segment_cluster"] = kmeans.fit_predict(X)

segment_summary = (
    customer_value.groupby("segment_cluster")
                  .agg(
                      customers=("customer_id", "nunique"),
                      avg_net_margin=("net_margin", "mean"),
                      avg_clv_proxy_12m=("clv_proxy_12m", "mean"),
                      avg_discount_rate=("discount_rate", "mean"),
                      avg_recency_days=("recency_days", "mean"),
                      avg_omnichannel=("omnichannel_flag", "mean"),
                  )
                  .round(3)
                  .reset_index()
)

# Etiquetado interpretativo simple
cluster_high_value = segment_summary["avg_clv_proxy_12m"].idxmax()
cluster_promo_dependent = segment_summary["avg_discount_rate"].idxmax()
remaining = set(segment_summary["segment_cluster"]) - {int(segment_summary.loc[cluster_high_value, "segment_cluster"]), int(segment_summary.loc[cluster_promo_dependent, "segment_cluster"])}
cluster_omni = (
    segment_summary[segment_summary["segment_cluster"].isin(remaining)]
    .sort_values("avg_omnichannel", ascending=False)
    .iloc[0]["segment_cluster"]
)
cluster_low_value = list(set(segment_summary["segment_cluster"]) - {
    int(segment_summary.loc[cluster_high_value, "segment_cluster"]),
    int(segment_summary.loc[cluster_promo_dependent, "segment_cluster"]),
    int(cluster_omni)
})[0]

segment_names = {
    int(segment_summary.loc[cluster_high_value, "segment_cluster"]): "Alta contribución sostenible",
    int(segment_summary.loc[cluster_promo_dependent, "segment_cluster"]): "Dependiente de promoción",
    int(cluster_omni): "Crecimiento omnicanal",
    int(cluster_low_value): "Reactivación selectiva",
}
customer_value["segmento_valor"] = customer_value["segment_cluster"].map(segment_names)
segment_summary["segmento_valor"] = segment_summary["segment_cluster"].map(segment_names)

segment_summary.sort_values("avg_clv_proxy_12m", ascending=False)


,segment_cluster,customers,avg_net_margin,avg_clv_proxy_12m,avg_discount_rate,avg_recency_days,avg_omnichannel,segmento_valor
0,0,10,9056.209,14140.109,0.084,70.818,0.909,Alta contribución sostenible
3,3,75,3558.871,3200.809,0.105,77.278,0.987,Crecimiento omnicanal
1,1,14,2834.371,2566.481,0.123,53.857,1.000,Dependiente de promoción
2,2,36,811.597,1923.892,0.083,234.286,0.056,Reactivación selectiva


In [4]:

acciones = {
    "Alta contribución sostenible": "Priorizar exclusividad, acceso anticipado y menor presión promocional.",
    "Crecimiento omnicanal": "Impulsar journeys de tienda + online, click&collect y cross-sell.",
    "Dependiente de promoción": "Limitar descuentos generalistas y rediseñar oferta con umbrales de margen.",
    "Reactivación selectiva": "Activar solo cuando exista señal de intención o afinidad clara."
}
customer_value["recomendacion_marketing"] = customer_value["segmento_valor"].map(acciones)

customer_value.sort_values(["clv_proxy_12m", "net_margin"], ascending=False).head(15)


,customer_id,gross_revenue,net_margin,discount_total,cac_total,expected_return_cost,tx_count,first_tx,last_tx,revenue_online,revenue_pos,coupons,redeemed,avg_coupon_discount,tier_fidelizacion,canal_preferido_declarado,consentimiento_marketing,omnichannel_flag,online_share,discount_rate,coupon_redemption_rate,recency_days,active_months,margin_per_tx,tx_per_year,retention_factor,clv_proxy_12m,segment_cluster,segmento_valor,recomendacion_marketing
9,C0010,5645.47,2442.43,577.48,25.53,240.42,7,2025-11-10 16:03:00,2025-12-02 06:20:00,1892.49,3752.98,0.0,0.0,0.000000,Oro,Tienda,0,1,0.335223,0.102291,NaN,11.0,1.000000,348.918571,84.000000,1.084061,31772.923499,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
84,C0082,16401.27,6514.59,1951.27,50.39,564.06,15,2025-05-11 22:06:47,2025-09-02 02:42:00,6894.91,9506.36,0.0,0.0,0.000000,Oro,NaN,no,1,0.420389,0.118971,NaN,102.0,3.717105,434.306000,48.424779,0.828908,17432.911158,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
70,C0069,14418.93,5433.67,414.47,77.08,1011.08,14,2025-05-04 18:49:00,2025-08-10 08:12:21,6785.16,7746.13,8.0,0.0,13.166667,Oro,Tienda,YES,1,0.466934,0.028745,0.0,125.0,3.190789,388.119286,52.651546,0.797474,16296.436224,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
21,C0021,18789.64,11097.29,357.91,87.87,537.88,14,2025-02-07 20:52:00,2025-12-10 02:55:00,10132.97,8656.67,11.0,0.0,12.111111,Oro,Mixto,YES,1,0.539285,0.019048,0.0,3.0,10.032895,792.663571,16.744918,1.135114,15066.465415,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
5,C0006,16481.18,8080.13,865.96,67.34,1708.61,15,2025-03-08 23:45:57,2025-11-05 12:15:18,9030.55,7450.63,0.0,0.0,0.000000,Oro,Online,0,1,0.547931,0.052542,NaN,38.0,7.927632,538.675333,22.705394,1.027501,12567.190947,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
81,C0079,10864.12,5199.18,2054.26,114.80,355.68,8,2025-01-10 07:34:23,2025-01-10 07:34:23,6891.78,3999.90,10.0,1.0,14.125000,NaN,Mixto,S,1,0.632756,0.189087,0.1,337.0,1.000000,649.897500,96.000000,0.200000,12478.032000,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
2,C0003,6343.99,1962.32,1103.32,37.65,208.40,7,2025-07-11 22:41:00,2025-09-01 11:51:00,3660.70,3412.43,8.0,0.0,12.625000,Plata,Mixto,YES,1,0.517550,0.173916,0.0,103.0,1.677632,280.331429,50.070588,0.806938,11326.467598,3,Crecimiento omnicanal,"Impulsar journeys de tienda + online, click&co..."
25,C0025,19748.90,8170.82,1362.25,64.28,1051.59,15,2025-02-04 00:49:00,2025-12-03 11:22:00,10362.84,9386.06,5.0,0.0,14.333333,Oro,Mixto,S,1,0.524730,0.068979,0.0,10.0,9.934211,544.721333,18.119205,1.098460,10841.712288,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
78,C0077,38647.11,18006.34,3730.27,103.48,2672.33,28,2024-02-01 20:56:26,2025-10-05 03:57:00,20818.81,17983.70,0.0,0.0,0.000000,BRONCE,Online,S,1,0.536533,0.096521,NaN,69.0,20.098684,643.083571,16.717512,0.927176,9967.849095,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."
79,C0077,38647.11,18006.34,3730.27,103.48,2672.33,28,2024-02-01 20:56:26,2025-10-05 03:57:00,20818.81,17983.70,0.0,0.0,0.000000,BRONCE,Online,NaN,1,0.536533,0.096521,NaN,69.0,20.098684,643.083571,16.717512,0.927176,9967.849095,0,Alta contribución sostenible,"Priorizar exclusividad, acceso anticipado y me..."



## Lectura de negocio

El objetivo no es encontrar “clientes que compran mucho”, sino detectar qué clientes:

- dejan margen neto después de costes,
- mantienen frecuencia razonable,
- no dependen excesivamente del descuento,
- y tienen potencial futuro activable.

Este enfoque permite reemplazar el RFM clásico por una segmentación más conectada con rentabilidad.


In [5]:

path_customers = mpu.export_csv(customer_value.sort_values(["clv_proxy_12m", "net_margin"], ascending=False), "nb10_customer_value_segmentation.csv")
path_summary = mpu.export_csv(segment_summary.sort_values("avg_clv_proxy_12m", ascending=False), "nb10_segment_summary.csv")

print("Guardado:", path_customers)
print("Guardado:", path_summary)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb10_customer_value_segmentation.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb10_segment_summary.csv
